# Primary experiments

Evaluate four feature scenarios with four classifiers under GridSearch and SCSO. Hyperparameters are selected by stratified five-fold cross-validation on the training partition. The test partition is held out until evaluation.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

DATA_FILE = PROJECT_ROOT / "data" / "gallstone.xlsx"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Download the UCI Gallstone dataset and save it to {DATA_FILE}")


In [ ]:
"""Primary experiment: four feature scenarios, four classifiers, and two optimizers.
All hyperparameters are tuned by stratified five-fold cross-validation.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier


RANDOM_SEED = 42

DATA_PATH = str(DATA_FILE)
TARGET_COL = "Gallstone Status"


POSITIVE_LABEL = 0  # verified: raw 0 = gallstone present; raw 1 = absent


SAMPLE_ID_COL = None

N_SPLITS = 5
TEST_SIZE = 0.20

HPO_BUDGET = 100
SCSO_POP_SIZE = 10
SCSO_S_MAX = 2.0

OUTPUT_DIR = RESULTS_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


SIGN_EPS = 1e-12


FEATURE_GROUPS = {
    "demographic": [
        "Age",
        "Gender",
        "Height",
        "Weight",
        "Body Mass Index (BMI)",
    ],
    "clinical": [
        "Comorbidity",
        "Coronary Artery Disease (CAD)",
        "Hypothyroidism",
        "Hyperlipidemia",
        "Diabetes Mellitus (DM)",
        "Hepatic Fat Accumulation (HFA)",
    ],
    "bioimpedance": [
        "Total Body Water (TBW)",
        "Extracellular Water (ECW)",
        "Intracellular Water (ICW)",
        "Extracellular Fluid/Total Body Water (ECF/TBW)",
        "Total Body Fat Ratio (TBFR) (%)",
        "Lean Mass (LM) (%)",
        "Body Protein Content (Protein) (%)",
        "Visceral Fat Rating (VFR)",
        "Bone Mass (BM)",
        "Muscle Mass (MM)",
        "Obesity (%)",
        "Total Fat Content (TFC)",
        "Visceral Fat Area (VFA)",
        "Visceral Muscle Area (VMA) (Kg)",
    ],
    "laboratory": [
        "Glucose",
        "Total Cholesterol (TC)",
        "Low Density Lipoprotein (LDL)",
        "High Density Lipoprotein (HDL)",
        "Triglyceride",
        "Aspartat Aminotransferaz (AST)",
        "Alanin Aminotransferaz (ALT)",
        "Alkaline Phosphatase (ALP)",
        "Creatinine",
        "Glomerular Filtration Rate (GFR)",
        "C-Reactive Protein (CRP)",
        "Hemoglobin (HGB)",
        "Vitamin D",
    ],
}


SCENARIOS = {
    "A_baseline": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
    ),
    "B_plus_bia": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
        + FEATURE_GROUPS["bioimpedance"]
    ),
    "C_plus_lab": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
        + FEATURE_GROUPS["laboratory"]
    ),
    "D_full": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
        + FEATURE_GROUPS["bioimpedance"]
        + FEATURE_GROUPS["laboratory"]
    ),
}


SCENARIO_ORDER = ["A_baseline", "B_plus_bia", "C_plus_lab", "D_full"]
MODEL_ORDER = ["LR", "SVM", "RF", "XGB"]
HPO_ORDER = ["GridSearch", "SCSO"]
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc"]


def make_pipeline(estimator) -> Pipeline:
    """Same preprocessing structure for every classifier."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", estimator),
        ]
    )


MODEL_SPECS = {
    "LR": {

        "estimator": LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_SEED,
        ),

        "grid": {
            "clf__C": np.logspace(-3, 2, 100),
        },

        "scso_spec": {
            "clf__C": (-3.0, 2.0, "log_cont"),
        },
    },
    "SVM": {
        "estimator": SVC(
            kernel="rbf",
            probability=False,
            random_state=RANDOM_SEED,
        ),

        "grid": {
            "clf__C": np.logspace(-2, 2, 10),
            "clf__gamma": np.logspace(-4, 0, 10),
        },

        "scso_spec": {
            "clf__C": (-2.0, 2.0, "log_cont"),
            "clf__gamma": (-4.0, 0.0, "log_cont"),
        },
    },
    "RF": {
        "estimator": RandomForestClassifier(
            criterion="gini",
            bootstrap=True,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),

        "grid": {
            "clf__n_estimators": [50, 100, 150, 225, 300],
            "clf__max_depth": [5, 10, 15, 20, 25],
            "clf__min_samples_split": [2, 5],
            "clf__min_samples_leaf": [1, 2],
        },
        "scso_spec": {
            "clf__n_estimators": (50.0, 300.0, "int"),
            "clf__max_depth": (5.0, 25.0, "int"),
            "clf__min_samples_split": (2.0, 5.0, "int"),
            "clf__min_samples_leaf": (1.0, 2.0, "int"),
        },
    },
    "XGB": {
        "estimator": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0,
        ),

        "grid": {
            "clf__n_estimators": [50, 100, 150, 225, 300],
            "clf__max_depth": [2, 3, 4, 6, 8],
            "clf__learning_rate": [0.01, 0.05, 0.10, 0.20],
        },
        "scso_spec": {
            "clf__n_estimators": (50.0, 300.0, "int"),
            "clf__max_depth": (2.0, 8.0, "int"),
            "clf__learning_rate": (0.01, 0.20, "cont"),
        },
    },
}


def to_builtin(value: Any) -> Any:
    """Convert numpy/pandas scalar values to JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if np.isscalar(value) else False:
        return None
    return value


def params_to_json(params: dict[str, Any]) -> str:
    return json.dumps(to_builtin(params), sort_keys=True)


def load_dataset(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Dataset not found: {p.resolve()}\n"
            f"Set DATA_PATH at the top of this script."
        )

    suffix = p.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(p)
    if suffix == ".csv":
        return pd.read_csv(p)

    raise ValueError(
        f"Unsupported file type '{suffix}'. Use .xlsx, .xls, or .csv."
    )


def validate_columns_and_data(df: pd.DataFrame) -> None:
    all_features = [c for cols in FEATURE_GROUPS.values() for c in cols]

    duplicated_feature_names = sorted(
        {c for c in all_features if all_features.count(c) > 1}
    )
    if duplicated_feature_names:
        raise ValueError(
            f"Duplicate feature names found across FEATURE_GROUPS: "
            f"{duplicated_feature_names}"
        )

    missing_features = [c for c in all_features if c not in df.columns]
    if missing_features:
        raise ValueError(
            "The following feature columns are missing from the dataset:\n"
            + "\n".join(f" - {c}" for c in missing_features)
            + "\n\nColumns actually found:\n"
            + "\n".join(f" - {c}" for c in df.columns)
        )

    if TARGET_COL not in df.columns:
        raise ValueError(
            f"TARGET_COL='{TARGET_COL}' was not found.\n"
            f"Available columns: {list(df.columns)}"
        )

    if SAMPLE_ID_COL is not None and SAMPLE_ID_COL not in df.columns:
        raise ValueError(
            f"SAMPLE_ID_COL='{SAMPLE_ID_COL}' was not found."
        )

    if df[TARGET_COL].isna().any():
        raise ValueError("Target column contains missing values.")

    feature_missing = df[all_features].isna().sum()
    feature_missing = feature_missing[feature_missing > 0]
    if len(feature_missing):
        raise ValueError(
            "Feature columns contain missing values, but the manuscript "
            "specifies no imputation. Missing counts:\n"
            + feature_missing.to_string()
        )


    non_numeric = [
        c for c in all_features
        if not pd.api.types.is_numeric_dtype(df[c])
    ]
    if non_numeric:
        raise TypeError(
            "These predictor columns are not numeric, while the current "
            "manuscript specifies no additional encoding:\n"
            + "\n".join(f" - {c}" for c in non_numeric)
        )

    expected_counts = {
        "demographic": 5,
        "clinical": 6,
        "bioimpedance": 14,
        "laboratory": 13,
    }
    for group, expected in expected_counts.items():
        actual = len(FEATURE_GROUPS[group])
        if actual != expected:
            raise ValueError(
                f"Feature group '{group}' has {actual} variables; "
                f"expected {expected}."
            )

    if len(set(all_features)) != 38:
        raise ValueError(
            f"Expected 38 unique predictors, found {len(set(all_features))}."
        )


def encode_binary_target(raw_target: pd.Series) -> tuple[pd.Series, dict[str, Any]]:
    values = list(pd.unique(raw_target))

    print("\nRaw target distribution:")
    print(raw_target.value_counts(dropna=False))

    if len(values) != 2:
        raise ValueError(
            f"Expected a binary target, found {len(values)} unique values: {values}"
        )

    if POSITIVE_LABEL is None:
        raise ValueError(
            "\nPOSITIVE_LABEL is still None.\n"
            "Set POSITIVE_LABEL at the top of the script to the RAW target value "
            "that means 'has gallstone', then rerun the experiment.\n"
            f"Observed raw labels: {values}"
        )

    if POSITIVE_LABEL not in values:
        raise ValueError(
            f"POSITIVE_LABEL={POSITIVE_LABEL!r} is not present in target values "
            f"{values}."
        )

    negative_values = [v for v in values if v != POSITIVE_LABEL]
    negative_label = negative_values[0]

    y = (raw_target == POSITIVE_LABEL).astype(int)

    mapping_info = {
        "raw_positive_label": to_builtin(POSITIVE_LABEL),
        "raw_negative_label": to_builtin(negative_label),
        "encoded_positive": 1,
        "encoded_negative": 0,
    }

    print("\nConfirmed target mapping:")
    print(f"  Positive class (1) = raw label {POSITIVE_LABEL!r}")
    print(f"  Negative class (0) = raw label {negative_label!r}")
    print("\nEncoded target distribution:")
    print(y.value_counts().sort_index())

    return y, mapping_info


def get_sample_ids(df: pd.DataFrame) -> pd.Series:
    if SAMPLE_ID_COL is not None:
        return df[SAMPLE_ID_COL].astype(str)
    return pd.Series(df.index.astype(str), index=df.index, name="sample_id")


def continuous_score(fitted_pipeline: Pipeline, X: pd.DataFrame) -> tuple[np.ndarray, str]:
    """
    Continuous score for ROC-AUC:
    - probability for LR/RF/XGB
    - decision_function for SVM
    """
    clf = fitted_pipeline.named_steps["clf"]

    if hasattr(clf, "predict_proba"):
        score = fitted_pipeline.predict_proba(X)[:, 1]
        return np.asarray(score, dtype=float), "probability"

    if hasattr(clf, "decision_function"):
        score = fitted_pipeline.decision_function(X)
        return np.asarray(score, dtype=float), "decision_function"

    raise AttributeError(
        f"Classifier {clf.__class__.__name__} provides neither "
        f"predict_proba nor decision_function."
    )


def compute_metrics(
    y_true: pd.Series | np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(
            precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y_true, y_score)),
    }


def summarize_fold_metrics(fold_df: pd.DataFrame) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    for metric in METRICS:
        vals = fold_df[metric].to_numpy(dtype=float)
        out[metric] = {
            "mean": float(np.mean(vals)),
            "sd": float(np.std(vals, ddof=1)),
        }
    return out


@dataclass
class CandidateEvaluation:
    eval_id: int
    params: dict[str, Any]
    fitness: float


class SCSO:
    """
    Sand Cat Swarm Optimization for maximizing CV Accuracy.

    Population update is synchronous and uses:
    - full population replacement after each update round,
    - no extra greedy acceptance rule,
    - random population member in exploration,
    - global best in exploitation,
    - roulette-wheel selection over 360 possible movement directions.

    Search coordinates are the encoded parameter domains themselves.
    """

    def __init__(
        self,
        param_spec: dict[str, tuple[float, float, str]],
        fitness_fn,
        pop_size: int = 10,
        max_evals: int = 100,
        s_max: float = 2.0,
        seed: int = RANDOM_SEED,
    ):
        if max_evals < pop_size:
            raise ValueError("max_evals must be >= pop_size.")
        if (max_evals - pop_size) % pop_size != 0:
            raise ValueError(
                "For an exact full-population budget, "
                "(max_evals - pop_size) must be divisible by pop_size."
            )

        self.param_spec = param_spec
        self.names = list(param_spec.keys())
        self.dim = len(self.names)

        self.lower = np.array(
            [param_spec[n][0] for n in self.names], dtype=float
        )
        self.upper = np.array(
            [param_spec[n][1] for n in self.names], dtype=float
        )

        self.fitness_fn = fitness_fn
        self.pop_size = int(pop_size)
        self.max_evals = int(max_evals)
        self.update_rounds = (self.max_evals - self.pop_size) // self.pop_size
        self.s_max = float(s_max)
        self.rng = np.random.default_rng(seed)

        self.n_evals = 0
        self.trace: list[CandidateEvaluation] = []
        self.best_history: list[dict[str, Any]] = []
        self.unique_param_keys: set[str] = set()


        self.angle_labels = np.arange(1, 361, dtype=float)
        self.angle_prob = self.angle_labels / self.angle_labels.sum()

    def _decode(self, x: np.ndarray) -> dict[str, Any]:
        params: dict[str, Any] = {}

        for i, name in enumerate(self.names):
            low, high, kind = self.param_spec[name]
            v = float(np.clip(x[i], low, high))

            if kind == "log_cont":
                decoded = float(10.0 ** v)
            elif kind == "int":
                decoded = int(np.clip(np.rint(v), math.ceil(low), math.floor(high)))
            elif kind == "cont":
                decoded = float(v)
            else:
                raise ValueError(f"Unknown parameter kind '{kind}' for {name}.")

            params[name] = decoded

        return params

    def _evaluate(self, x: np.ndarray) -> float:
        if self.n_evals >= self.max_evals:
            raise RuntimeError("SCSO evaluation budget exceeded.")

        params = self._decode(x)
        fitness = float(self.fitness_fn(params))

        self.n_evals += 1
        self.unique_param_keys.add(params_to_json(params))
        self.trace.append(
            CandidateEvaluation(
                eval_id=self.n_evals,
                params=params,
                fitness=fitness,
            )
        )
        return fitness

    def _evaluate_population(self, pop: np.ndarray) -> np.ndarray:
        return np.array([self._evaluate(pop[i]) for i in range(self.pop_size)])

    def _roulette_angle(self) -> float:
        """
        Roulette-wheel selection among 360 direction labels.

        The selected label is used directly in cos(alpha), matching the
        original-style SCSO code family that applies cos() to the roulette
        output rather than introducing a separate model-specific parameter.
        """
        return float(self.rng.choice(self.angle_labels, p=self.angle_prob))

    def _record_best(
        self,
        round_id: int,
        fitness: np.ndarray,
        pop: np.ndarray,
        best_fit: float,
        best_pos: np.ndarray,
    ) -> tuple[float, np.ndarray]:
        idx = int(np.argmax(fitness))
        if fitness[idx] > best_fit:
            best_fit = float(fitness[idx])
            best_pos = pop[idx].copy()

        self.best_history.append(
            {
                "round": int(round_id),
                "n_evals": int(self.n_evals),
                "best_cv_accuracy": float(best_fit),
            }
        )
        return best_fit, best_pos

    def run(self) -> dict[str, Any]:

        pop = self.rng.uniform(
            low=self.lower,
            high=self.upper,
            size=(self.pop_size, self.dim),
        )


        fitness = self._evaluate_population(pop)
        best_idx = int(np.argmax(fitness))
        best_fit = float(fitness[best_idx])
        best_pos = pop[best_idx].copy()
        self.best_history.append(
            {
                "round": 0,
                "n_evals": int(self.n_evals),
                "best_cv_accuracy": float(best_fit),
            }
        )


        for t in range(self.update_rounds):


            T = max(1, self.update_rounds)
            r_G = self.s_max - (self.s_max * t / T)

            old_pop = pop.copy()
            new_pop = np.empty_like(old_pop)

            for i in range(self.pop_size):
                r = r_G * self.rng.random()
                R = 2.0 * r_G * self.rng.random() - r_G


                current = old_pop[i]
                updated = current.copy()

                for j in range(self.dim):
                    alpha = self._roulette_angle()

                    if abs(R) <= 1.0:

                        x_rand = abs(
                            self.rng.random() * best_pos[j] - current[j]
                        )
                        updated[j] = (
                            best_pos[j]
                            - r * x_rand * math.cos(alpha)
                        )
                    else:


                        cp = int(self.rng.integers(0, self.pop_size))
                        candidate = old_pop[cp, j]
                        updated[j] = (
                            r * (
                                candidate
                                - self.rng.random() * current[j]
                            )
                        )

                new_pop[i] = np.clip(updated, self.lower, self.upper)


            pop = new_pop
            fitness = self._evaluate_population(pop)

            best_fit, best_pos = self._record_best(
                round_id=t + 1,
                fitness=fitness,
                pop=pop,
                best_fit=best_fit,
                best_pos=best_pos,
            )

        if self.n_evals != self.max_evals:
            raise RuntimeError(
                f"SCSO ended with {self.n_evals} evaluations; "
                f"expected exactly {self.max_evals}."
            )

        return {
            "best_params": self._decode(best_pos),
            "best_cv_accuracy": float(best_fit),
            "n_evals": int(self.n_evals),
            "n_unique_configs": int(len(self.unique_param_keys)),
            "history": self.best_history,
            "trace": [
                {
                    "eval_id": e.eval_id,
                    "params": e.params,
                    "cv_accuracy": e.fitness,
                }
                for e in self.trace
            ],
        }




In [ ]:
def make_cv_accuracy_objective(
    base_pipe: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
):
    """Mean Stratified 5-Fold CV Accuracy — common HPO objective."""
    def objective(params: dict[str, Any]) -> float:
        scores = []

        for train_idx, val_idx in cv_splits:
            pipe = clone(base_pipe)
            pipe.set_params(**params)

            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            pipe.fit(X_tr, y_tr)
            pred = pipe.predict(X_val)
            scores.append(accuracy_score(y_val, pred))

        return float(np.mean(scores))

    return objective


def evaluate_best_params_on_cv(
    base_pipe: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
    best_params: dict[str, Any],
    scenario: str,
    model_name: str,
    hpo_name: str,
) -> tuple[dict[str, dict[str, float]], pd.DataFrame]:
    """Return CV mean±SD plus the raw metric row for every fold."""
    fold_rows = []

    for fold_id, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        pipe = clone(base_pipe)
        pipe.set_params(**best_params)

        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        pipe.fit(X_tr, y_tr)

        pred = pipe.predict(X_val)
        score, score_type = continuous_score(pipe, X_val)
        m = compute_metrics(y_val, pred, score)

        fold_rows.append(
            {
                "scenario": scenario,
                "model": model_name,
                "hpo": hpo_name,
                "fold": fold_id,
                "n_train_fold": len(train_idx),
                "n_validation_fold": len(val_idx),
                "score_type": score_type,
                **m,
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    summary = summarize_fold_metrics(fold_df)
    return summary, fold_df


def evaluate_on_heldout_test(
    base_pipe: Pipeline,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    sample_ids_test: pd.Series,
    best_params: dict[str, Any],
    scenario: str,
    model_name: str,
    hpo_name: str,
) -> tuple[dict[str, float], pd.DataFrame]:
    """Refit on full training set once, then evaluate held-out test data."""
    pipe = clone(base_pipe)
    pipe.set_params(**best_params)
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    score, score_type = continuous_score(pipe, X_test)
    metrics = compute_metrics(y_test, pred, score)

    predictions = pd.DataFrame(
        {
            "sample_id": sample_ids_test.astype(str).to_numpy(),
            "scenario": scenario,
            "model": model_name,
            "hpo": hpo_name,
            "true_label": y_test.to_numpy(dtype=int),
            "predicted_label": np.asarray(pred, dtype=int),
            "score": np.asarray(score, dtype=float),
            "score_type": score_type,
        }
    )

    return metrics, predictions


def run_gridsearch(
    base_pipe: Pipeline,
    grid: dict[str, list[Any] | np.ndarray],
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
) -> dict[str, Any]:
    gs = GridSearchCV(
        estimator=base_pipe,
        param_grid=grid,
        scoring="accuracy",
        cv=cv_splits,
        n_jobs=-1,
        refit=False,
        return_train_score=False,
    )
    gs.fit(X, y)

    trace_rows = []
    params_list = gs.cv_results_["params"]
    mean_scores = gs.cv_results_["mean_test_score"]
    rank_scores = gs.cv_results_["rank_test_score"]

    for i, (params, mean_score, rank) in enumerate(
        zip(params_list, mean_scores, rank_scores),
        start=1,
    ):
        trace_rows.append(
            {
                "eval_id": i,
                "params": to_builtin(params),
                "cv_accuracy": float(mean_score),
                "rank": int(rank),
            }
        )

    return {
        "best_params": to_builtin(gs.best_params_),
        "best_cv_accuracy": float(gs.best_score_),
        "n_evals": int(len(params_list)),
        "n_unique_configs": int(
            len({params_to_json(p) for p in params_list})
        ),
        "trace": trace_rows,
    }


def run_scso(
    base_pipe: Pipeline,
    scso_spec: dict[str, tuple[float, float, str]],
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
) -> dict[str, Any]:
    objective = make_cv_accuracy_objective(base_pipe, X, y, cv_splits)

    optimizer = SCSO(
        param_spec=scso_spec,
        fitness_fn=objective,
        pop_size=SCSO_POP_SIZE,
        max_evals=HPO_BUDGET,
        s_max=SCSO_S_MAX,
        seed=RANDOM_SEED,
    )
    return optimizer.run()


CONTRASTS = {
    "B-A_Bioimpedance": ("B_plus_bia", "A_baseline"),
    "C-A_Laboratory": ("C_plus_lab", "A_baseline"),
    "D-B_Full_given_BIA": ("D_full", "B_plus_bia"),
    "D-C_Full_given_Lab": ("D_full", "C_plus_lab"),
}


def sign_label(value: float, eps: float = SIGN_EPS) -> str:
    if value > eps:
        return "positive"
    if value < -eps:
        return "negative"
    return "zero"




In [ ]:
def build_contribution_deltas(
    results_df: pd.DataFrame,
    stage: str,
) -> pd.DataFrame:
    """
    stage='test' uses test_* metrics.
    stage='cv' uses cv_*_mean metrics.
    """
    if stage not in {"test", "cv"}:
        raise ValueError("stage must be 'test' or 'cv'.")

    rows = []

    for model_name in MODEL_ORDER:
        for hpo_name in HPO_ORDER:
            sub = results_df[
                (results_df["model"] == model_name)
                & (results_df["hpo"] == hpo_name)
            ].set_index("scenario")

            for metric in METRICS:
                value_col = (
                    f"test_{metric}"
                    if stage == "test"
                    else f"cv_{metric}_mean"
                )

                for contrast_name, (higher_scenario, reference_scenario) in CONTRASTS.items():
                    higher = float(sub.loc[higher_scenario, value_col])
                    reference = float(sub.loc[reference_scenario, value_col])
                    delta = higher - reference

                    rows.append(
                        {
                            "stage": stage,
                            "model": model_name,
                            "hpo": hpo_name,
                            "metric": metric,
                            "contrast": contrast_name,
                            "higher_scenario": higher_scenario,
                            "reference_scenario": reference_scenario,
                            "higher_value": higher,
                            "reference_value": reference,
                            "delta": delta,
                            "direction": sign_label(delta),
                        }
                    )

    return pd.DataFrame(rows)


def build_classifier_consistency(
    delta_test: pd.DataFrame,
    delta_cv: pd.DataFrame,
) -> pd.DataFrame:
    combined = pd.concat([delta_cv, delta_test], ignore_index=True)
    rows = []

    for (stage, hpo, metric, contrast), group in combined.groupby(
        ["stage", "hpo", "metric", "contrast"],
        sort=False,
    ):
        deltas = group["delta"].astype(float)

        rows.append(
            {
                "stage": stage,
                "hpo": hpo,
                "metric": metric,
                "contrast": contrast,
                "positive_models": int((deltas > SIGN_EPS).sum()),
                "zero_models": int((deltas.abs() <= SIGN_EPS).sum()),
                "negative_models": int((deltas < -SIGN_EPS).sum()),
                "n_models": int(len(deltas)),
                "mean_delta": float(deltas.mean()),
                "median_delta": float(deltas.median()),
                "min_delta": float(deltas.min()),
                "max_delta": float(deltas.max()),
            }
        )

    return pd.DataFrame(rows)


def build_hpo_comparison(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compare SCSO vs GridSearch for every model, scenario, metric,
    using both CV mean and held-out test performance.
    """
    rows = []

    for scenario in SCENARIO_ORDER:
        for model_name in MODEL_ORDER:
            sub = results_df[
                (results_df["scenario"] == scenario)
                & (results_df["model"] == model_name)
            ].set_index("hpo")

            for metric in METRICS:
                for stage in ["cv", "test"]:
                    col = (
                        f"cv_{metric}_mean"
                        if stage == "cv"
                        else f"test_{metric}"
                    )
                    gs = float(sub.loc["GridSearch", col])
                    scso = float(sub.loc["SCSO", col])
                    delta = scso - gs

                    rows.append(
                        {
                            "stage": stage,
                            "scenario": scenario,
                            "model": model_name,
                            "metric": metric,
                            "gridsearch_value": gs,
                            "scso_value": scso,
                            "delta_hpo_scso_minus_grid": delta,
                            "direction": sign_label(delta),
                        }
                    )

    return pd.DataFrame(rows)


def build_hpo_feature_delta_consistency(
    delta_test: pd.DataFrame,
    delta_cv: pd.DataFrame,
) -> pd.DataFrame:
    """
    Primary RQ3 support:
    Does the direction of each feature-group delta stay the same when
    HPO changes from GridSearch to SCSO?
    """
    combined = pd.concat([delta_cv, delta_test], ignore_index=True)
    rows = []

    for stage in ["cv", "test"]:
        stage_df = combined[combined["stage"] == stage]

        for model_name in MODEL_ORDER:
            for metric in METRICS:
                for contrast in CONTRASTS:
                    sub = stage_df[
                        (stage_df["model"] == model_name)
                        & (stage_df["metric"] == metric)
                        & (stage_df["contrast"] == contrast)
                    ].set_index("hpo")

                    gs_delta = float(sub.loc["GridSearch", "delta"])
                    scso_delta = float(sub.loc["SCSO", "delta"])
                    gs_dir = sign_label(gs_delta)
                    scso_dir = sign_label(scso_delta)

                    rows.append(
                        {
                            "stage": stage,
                            "model": model_name,
                            "metric": metric,
                            "contrast": contrast,
                            "gridsearch_delta": gs_delta,
                            "scso_delta": scso_delta,
                            "gridsearch_direction": gs_dir,
                            "scso_direction": scso_dir,
                            "same_direction": bool(gs_dir == scso_dir),
                        }
                    )

    return pd.DataFrame(rows)


def export_excel_bundle(
    results_df: pd.DataFrame,
    fold_df: pd.DataFrame,
    best_params_df: pd.DataFrame,
    test_pred_df: pd.DataFrame,
    delta_test_df: pd.DataFrame,
    delta_cv_df: pd.DataFrame,
    consistency_df: pd.DataFrame,
    hpo_compare_df: pd.DataFrame,
    hpo_delta_consistency_df: pd.DataFrame,
    scso_history_df: pd.DataFrame,
    search_trace_df: pd.DataFrame,
) -> None:
    path = OUTPUT_DIR / "gallstone_results_bundle.xlsx"

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        results_df.to_excel(writer, sheet_name="Results32", index=False)
        fold_df.to_excel(writer, sheet_name="FoldMetrics", index=False)
        best_params_df.to_excel(writer, sheet_name="BestParams", index=False)
        test_pred_df.to_excel(writer, sheet_name="TestPredictions", index=False)
        delta_test_df.to_excel(writer, sheet_name="DeltaTest", index=False)
        delta_cv_df.to_excel(writer, sheet_name="DeltaCV", index=False)
        consistency_df.to_excel(writer, sheet_name="Consistency", index=False)
        hpo_compare_df.to_excel(writer, sheet_name="HPOCompare", index=False)
        hpo_delta_consistency_df.to_excel(
            writer,
            sheet_name="HPODeltaConsistency",
            index=False,
        )
        scso_history_df.to_excel(
            writer,
            sheet_name="SCSOHistory",
            index=False,
        )
        search_trace_df.to_excel(
            writer,
            sheet_name="SearchTrace",
            index=False,
        )




In [ ]:
def main() -> None:
    np.random.seed(RANDOM_SEED)

    df = load_dataset(DATA_PATH)
    print(f"Dataset loaded: {DATA_PATH}")
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}): {list(df.columns)}")

    validate_columns_and_data(df)

    y_full, target_mapping = encode_binary_target(df[TARGET_COL])
    sample_ids_full = get_sample_ids(df)

    if len(df) != 319:
        print(
            f"\nWARNING: manuscript currently describes 319 observations, "
            f"but this file contains {len(df)} rows. Verify before using results."
        )


    all_idx = np.arange(len(df))

    train_idx_global, test_idx_global = train_test_split(
        all_idx,
        test_size=TEST_SIZE,
        stratify=y_full,
        random_state=RANDOM_SEED,
    )

    y_train_global = y_full.iloc[train_idx_global].reset_index(drop=True)
    y_test_global = y_full.iloc[test_idx_global].reset_index(drop=True)

    sample_ids_test = (
        sample_ids_full.iloc[test_idx_global]
        .reset_index(drop=True)
    )


    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )
    cv_splits = list(
        skf.split(
            np.zeros(len(y_train_global)),
            y_train_global,
        )
    )

    print("\nGlobal split:")
    print(f"  Training n = {len(train_idx_global)}")
    print(f"  Test n     = {len(test_idx_global)}")
    print("\nTraining class counts:")
    print(y_train_global.value_counts().sort_index())
    print("\nTest class counts:")
    print(y_test_global.value_counts().sort_index())

    results_records: list[dict[str, Any]] = []
    fold_frames: list[pd.DataFrame] = []
    prediction_frames: list[pd.DataFrame] = []
    best_param_rows: list[dict[str, Any]] = []
    scso_history_rows: list[dict[str, Any]] = []
    search_trace_rows: list[dict[str, Any]] = []
    raw_results: list[dict[str, Any]] = []

    total_runs = len(SCENARIO_ORDER) * len(MODEL_ORDER) * len(HPO_ORDER)
    completed = 0

    for scenario in SCENARIO_ORDER:
        cols = SCENARIOS[scenario]

        X_full = df[cols]
        X_train = (
            X_full.iloc[train_idx_global]
            .reset_index(drop=True)
        )
        X_test = (
            X_full.iloc[test_idx_global]
            .reset_index(drop=True)
        )

        for model_name in MODEL_ORDER:
            spec = MODEL_SPECS[model_name]
            base_pipe = make_pipeline(clone(spec["estimator"]))


            t0 = time.perf_counter()
            gs = run_gridsearch(
                base_pipe=base_pipe,
                grid=spec["grid"],
                X=X_train,
                y=y_train_global,
                cv_splits=cv_splits,
            )
            gs_search_time = time.perf_counter() - t0

            gs_cv_summary, gs_fold_df = evaluate_best_params_on_cv(
                base_pipe=base_pipe,
                X=X_train,
                y=y_train_global,
                cv_splits=cv_splits,
                best_params=gs["best_params"],
                scenario=scenario,
                model_name=model_name,
                hpo_name="GridSearch",
            )

            gs_test_metrics, gs_pred_df = evaluate_on_heldout_test(
                base_pipe=base_pipe,
                X_train=X_train,
                y_train=y_train_global,
                X_test=X_test,
                y_test=y_test_global,
                sample_ids_test=sample_ids_test,
                best_params=gs["best_params"],
                scenario=scenario,
                model_name=model_name,
                hpo_name="GridSearch",
            )

            gs_record = {
                "scenario": scenario,
                "n_features": len(cols),
                "model": model_name,
                "hpo": "GridSearch",
                "objective_cv_accuracy": gs["best_cv_accuracy"],
                "n_evals": gs["n_evals"],
                "n_unique_configs": gs["n_unique_configs"],
                "search_time_sec": gs_search_time,
                "best_params_json": params_to_json(gs["best_params"]),
            }

            for metric in METRICS:
                gs_record[f"cv_{metric}_mean"] = gs_cv_summary[metric]["mean"]
                gs_record[f"cv_{metric}_sd"] = gs_cv_summary[metric]["sd"]
                gs_record[f"test_{metric}"] = gs_test_metrics[metric]

            results_records.append(gs_record)
            fold_frames.append(gs_fold_df)
            prediction_frames.append(gs_pred_df)

            best_param_rows.append(
                {
                    "scenario": scenario,
                    "model": model_name,
                    "hpo": "GridSearch",
                    "best_params_json": params_to_json(gs["best_params"]),
                    "objective_cv_accuracy": gs["best_cv_accuracy"],
                    "n_evals": gs["n_evals"],
                    "n_unique_configs": gs["n_unique_configs"],
                }
            )

            for row in gs["trace"]:
                search_trace_rows.append(
                    {
                        "scenario": scenario,
                        "model": model_name,
                        "hpo": "GridSearch",
                        "eval_id": row["eval_id"],
                        "cv_accuracy": row["cv_accuracy"],
                        "rank": row.get("rank"),
                        "params_json": params_to_json(row["params"]),
                    }
                )

            raw_results.append(
                {
                    "scenario": scenario,
                    "model": model_name,
                    "hpo": "GridSearch",
                    "best_params": gs["best_params"],
                    "objective_cv_accuracy": gs["best_cv_accuracy"],
                    "n_evals": gs["n_evals"],
                    "n_unique_configs": gs["n_unique_configs"],
                    "search_time_sec": gs_search_time,
                    "cv_summary": gs_cv_summary,
                    "test_metrics": gs_test_metrics,
                }
            )

            completed += 1
            print(
                f"[{completed:02d}/{total_runs}] "
                f"{scenario} | {model_name} | GridSearch | "
                f"CV Acc={gs['best_cv_accuracy']:.4f} | "
                f"Test Acc={gs_test_metrics['accuracy']:.4f}"
            )


            t0 = time.perf_counter()
            scso = run_scso(
                base_pipe=base_pipe,
                scso_spec=spec["scso_spec"],
                X=X_train,
                y=y_train_global,
                cv_splits=cv_splits,
            )
            scso_search_time = time.perf_counter() - t0

            scso_cv_summary, scso_fold_df = evaluate_best_params_on_cv(
                base_pipe=base_pipe,
                X=X_train,
                y=y_train_global,
                cv_splits=cv_splits,
                best_params=scso["best_params"],
                scenario=scenario,
                model_name=model_name,
                hpo_name="SCSO",
            )

            scso_test_metrics, scso_pred_df = evaluate_on_heldout_test(
                base_pipe=base_pipe,
                X_train=X_train,
                y_train=y_train_global,
                X_test=X_test,
                y_test=y_test_global,
                sample_ids_test=sample_ids_test,
                best_params=scso["best_params"],
                scenario=scenario,
                model_name=model_name,
                hpo_name="SCSO",
            )

            scso_record = {
                "scenario": scenario,
                "n_features": len(cols),
                "model": model_name,
                "hpo": "SCSO",
                "objective_cv_accuracy": scso["best_cv_accuracy"],
                "n_evals": scso["n_evals"],
                "n_unique_configs": scso["n_unique_configs"],
                "search_time_sec": scso_search_time,
                "best_params_json": params_to_json(scso["best_params"]),
            }

            for metric in METRICS:
                scso_record[f"cv_{metric}_mean"] = scso_cv_summary[metric]["mean"]
                scso_record[f"cv_{metric}_sd"] = scso_cv_summary[metric]["sd"]
                scso_record[f"test_{metric}"] = scso_test_metrics[metric]

            results_records.append(scso_record)
            fold_frames.append(scso_fold_df)
            prediction_frames.append(scso_pred_df)

            best_param_rows.append(
                {
                    "scenario": scenario,
                    "model": model_name,
                    "hpo": "SCSO",
                    "best_params_json": params_to_json(scso["best_params"]),
                    "objective_cv_accuracy": scso["best_cv_accuracy"],
                    "n_evals": scso["n_evals"],
                    "n_unique_configs": scso["n_unique_configs"],
                }
            )

            for row in scso["history"]:
                scso_history_rows.append(
                    {
                        "scenario": scenario,
                        "model": model_name,
                        "hpo": "SCSO",
                        **row,
                    }
                )

            for row in scso["trace"]:
                search_trace_rows.append(
                    {
                        "scenario": scenario,
                        "model": model_name,
                        "hpo": "SCSO",
                        "eval_id": row["eval_id"],
                        "cv_accuracy": row["cv_accuracy"],
                        "rank": np.nan,
                        "params_json": params_to_json(row["params"]),
                    }
                )

            raw_results.append(
                {
                    "scenario": scenario,
                    "model": model_name,
                    "hpo": "SCSO",
                    "best_params": scso["best_params"],
                    "objective_cv_accuracy": scso["best_cv_accuracy"],
                    "n_evals": scso["n_evals"],
                    "n_unique_configs": scso["n_unique_configs"],
                    "search_time_sec": scso_search_time,
                    "cv_summary": scso_cv_summary,
                    "test_metrics": scso_test_metrics,
                    "history": scso["history"],
                }
            )

            completed += 1
            print(
                f"[{completed:02d}/{total_runs}] "
                f"{scenario} | {model_name} | SCSO | "
                f"CV Acc={scso['best_cv_accuracy']:.4f} | "
                f"Test Acc={scso_test_metrics['accuracy']:.4f} | "
                f"Unique configs={scso['n_unique_configs']}/{scso['n_evals']}"
            )


    results_df = pd.DataFrame(results_records)

    scenario_rank = {v: i for i, v in enumerate(SCENARIO_ORDER)}
    model_rank = {v: i for i, v in enumerate(MODEL_ORDER)}
    hpo_rank = {v: i for i, v in enumerate(HPO_ORDER)}

    results_df["_scenario_order"] = results_df["scenario"].map(scenario_rank)
    results_df["_model_order"] = results_df["model"].map(model_rank)
    results_df["_hpo_order"] = results_df["hpo"].map(hpo_rank)

    results_df = (
        results_df
        .sort_values(["_scenario_order", "_model_order", "_hpo_order"])
        .drop(columns=["_scenario_order", "_model_order", "_hpo_order"])
        .reset_index(drop=True)
    )

    fold_df = pd.concat(fold_frames, ignore_index=True)
    test_pred_df = pd.concat(prediction_frames, ignore_index=True)
    best_params_df = pd.DataFrame(best_param_rows)
    scso_history_df = pd.DataFrame(scso_history_rows)
    search_trace_df = pd.DataFrame(search_trace_rows)


    delta_test_df = build_contribution_deltas(results_df, stage="test")
    delta_cv_df = build_contribution_deltas(results_df, stage="cv")

    classifier_consistency_df = build_classifier_consistency(
        delta_test=delta_test_df,
        delta_cv=delta_cv_df,
    )

    hpo_compare_df = build_hpo_comparison(results_df)

    hpo_delta_consistency_df = build_hpo_feature_delta_consistency(
        delta_test=delta_test_df,
        delta_cv=delta_cv_df,
    )


    results_df.to_csv(
        OUTPUT_DIR / "results_32_combinations.csv",
        index=False,
    )
    fold_df.to_csv(
        OUTPUT_DIR / "fold_metrics.csv",
        index=False,
    )
    best_params_df.to_csv(
        OUTPUT_DIR / "best_hyperparameters.csv",
        index=False,
    )
    test_pred_df.to_csv(
        OUTPUT_DIR / "test_predictions.csv",
        index=False,
    )
    delta_test_df.to_csv(
        OUTPUT_DIR / "contribution_deltas_test.csv",
        index=False,
    )
    delta_cv_df.to_csv(
        OUTPUT_DIR / "contribution_deltas_cv.csv",
        index=False,
    )
    classifier_consistency_df.to_csv(
        OUTPUT_DIR / "classifier_consistency.csv",
        index=False,
    )
    hpo_compare_df.to_csv(
        OUTPUT_DIR / "hpo_comparison.csv",
        index=False,
    )
    hpo_delta_consistency_df.to_csv(
        OUTPUT_DIR / "hpo_feature_delta_consistency.csv",
        index=False,
    )
    scso_history_df.to_csv(
        OUTPUT_DIR / "scso_history.csv",
        index=False,
    )
    search_trace_df.to_csv(
        OUTPUT_DIR / "search_trace.csv",
        index=False,
    )


    with open(
        OUTPUT_DIR / "raw_results.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            to_builtin(raw_results),
            f,
            indent=2,
            ensure_ascii=False,
        )

    metadata = {
        "data_path": str(DATA_PATH),
        "n_rows": int(len(df)),
        "n_predictors": 38,
        "target_col": TARGET_COL,
        "target_mapping": target_mapping,
        "random_seed": RANDOM_SEED,
        "test_size": TEST_SIZE,
        "n_splits": N_SPLITS,
        "hpo_budget": HPO_BUDGET,
        "scso_pop_size": SCSO_POP_SIZE,
        "scso_update_rounds": int(
            (HPO_BUDGET - SCSO_POP_SIZE) // SCSO_POP_SIZE
        ),
        "train_indices_original": train_idx_global.tolist(),
        "test_indices_original": test_idx_global.tolist(),
        "scenario_feature_counts": {
            name: len(cols)
            for name, cols in SCENARIOS.items()
        },
        "train_class_counts": {
            str(k): int(v)
            for k, v in y_train_global.value_counts().sort_index().items()
        },
        "test_class_counts": {
            str(k): int(v)
            for k, v in y_test_global.value_counts().sort_index().items()
        },
    }

    with open(
        OUTPUT_DIR / "run_metadata.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            to_builtin(metadata),
            f,
            indent=2,
            ensure_ascii=False,
        )


    export_excel_bundle(
        results_df=results_df,
        fold_df=fold_df,
        best_params_df=best_params_df,
        test_pred_df=test_pred_df,
        delta_test_df=delta_test_df,
        delta_cv_df=delta_cv_df,
        consistency_df=classifier_consistency_df,
        hpo_compare_df=hpo_compare_df,
        hpo_delta_consistency_df=hpo_delta_consistency_df,
        scso_history_df=scso_history_df,
        search_trace_df=search_trace_df,
    )

    print(f"Primary experiment finished: {len(results_df)} configurations")
    print(f"Results saved to {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()
